# Projeto Final: Classificar Produtos de Resíduos Usando Transfer Learning
Este notebook demonstra a criação de um modelo de classificação de resíduos usando VGG16 pré-treinado, incluindo etapas de pré-processamento, treinamento e visualização de resultados.

In [ ]:
import os
import pathlib
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers

print('TensorFlow version:', tf.__version__)

In [ ]:
base_path = pathlib.Path('data')
train_dir = base_path / 'train'
val_dir = base_path / 'val'
test_dir = base_path / 'test'
categories = ['reciclavel', 'organico']
image_size = (224, 224)
num_samples_per_class = 10
val_samples_per_class = 4
test_samples_per_class = 4
np.random.seed(42)

for folder in [train_dir, val_dir, test_dir]:
    for category in categories:
        path = folder / category
        path.mkdir(parents=True, exist_ok=True)

def create_images(folder, category, count, seed):
    rng = np.random.RandomState(seed)
    for i in range(count):
        array = rng.randint(0, 256, size=(*image_size, 3), dtype=np.uint8)
        image = Image.fromarray(array)
        image.save(folder / category / f'{category}_{i}.png')

create_images(train_dir, 'reciclavel', num_samples_per_class, 1)
create_images(train_dir, 'organico', num_samples_per_class, 2)
create_images(val_dir, 'reciclavel', val_samples_per_class, 3)
create_images(val_dir, 'organico', val_samples_per_class, 4)
create_images(test_dir, 'reciclavel', test_samples_per_class, 5)
create_images(test_dir, 'organico', test_samples_per_class, 6)

print('Data directories created:')
print(' -', train_dir)
print(' -', val_dir)
print(' -', test_dir)

In [ ]:
batch_size = 4

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=True
)
validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=image_size,
    batch_size=1,
    class_mode='binary',
    shuffle=False
)

print('train_generator length:', len(train_generator))

In [ ]:
def build_vgg16_model(trainable=False):
    base_model = VGG16(include_top=False, weights='imagenet', input_shape=(*image_size, 3), pooling='avg')
    base_model.trainable = trainable
    if trainable:
        for layer in base_model.layers[:-4]:
            layer.trainable = False
    inputs = layers.Input(shape=(*image_size, 3))
    x = base_model(inputs, training=False)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs, outputs)

extract_feat_model = build_vgg16_model(trainable=False)
extract_feat_model.summary()

In [ ]:
extract_feat_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print('Model compiled successfully.')

In [ ]:
history_extract = extract_feat_model.fit(
    train_generator,
    epochs=3,
    validation_data=validation_generator,
    verbose=2
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_extract.history['accuracy'], label='train_accuracy')
plt.plot(history_extract.history['val_accuracy'], label='val_accuracy')
plt.title('Curvas de Precisão - Modelo de Extração de Recursos')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
fine_tune_model = build_vgg16_model(trainable=True)
fine_tune_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
history_fine = fine_tune_model.fit(
    train_generator,
    epochs=3,
    validation_data=validation_generator,
    verbose=2
)

plt.figure(figsize=(8, 5))
plt.plot(history_fine.history['loss'], label='train_loss')
plt.plot(history_fine.history['val_loss'], label='val_loss')
plt.title('Curvas de Perda - Modelo Ajustado')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_fine.history['accuracy'], label='train_accuracy')
plt.plot(history_fine.history['val_accuracy'], label='val_accuracy')
plt.title('Curvas de Precisão - Modelo Ajustado')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
index_to_plot = 1
class_names = {v: k for k, v in test_generator.class_indices.items()}

def display_prediction(model, generator, index, title):
    images, labels = generator[index]
    image = images[0]
    true_label = class_names[int(labels[0])]
    prediction = model.predict(np.expand_dims(image, axis=0), verbose=0)[0][0]
    predicted_label = class_names[int(np.round(prediction))]
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'{title}
True: {true_label} | Predito: {predicted_label} ({prediction:.2f})')
    plt.show()

display_prediction(extract_feat_model, test_generator, index_to_plot, 'Teste - Modelo de Extração de Recursos')
display_prediction(fine_tune_model, test_generator, index_to_plot, 'Teste - Modelo Ajustado')